In [28]:
import torch
from mininn import (
    Conv2D, ReLU, MaxPool2D, Flatten,
    Linear, Sequential, ResidualBlock,Adam
)
from mininn.training import train_one_epoch, evaluate
from torchvision import datasets, transforms
from copy import copy
from torch.utils.data import DataLoader, random_split, Subset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
g = torch.Generator(device=device).manual_seed(42)

data_root = "../data_1"
batch_size = 64

In [29]:
model = Sequential([
    Conv2D(3, 8, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    Conv2D(8, 16, 3, padding=1, device=device, generator=g),
    ReLU(),
    MaxPool2D(2, 2),

    ResidualBlock(16, device=device, generator=g),
    Flatten(),
    Linear(1024, 10, device=device, generator=g),
])

images = torch.randn(
    (2, 3, 32, 32), device=device, generator=g
)
labels = torch.tensor([0, 1], device=device)

model.train()
logits = model(images)
loss = torch.nn.functional.cross_entropy(logits, labels)
loss.backward()

print("Output shape:", logits.shape)
print("Loss:", loss.item())
print("All parameter gradients present:", all(
    p.grad is not None for p in model.parameters()
))
print("All parameter gradients finite:", all(
    p.grad is not None and torch.isfinite(p.grad).all().item()
    for p in model.parameters()
))

Output shape: torch.Size([2, 10])
Loss: 7.386048793792725
All parameter gradients present: True
All parameter gradients finite: True


In [22]:
w = torch.tensor([2.0], device=device, requires_grad=True)
optimizer = Adam([w], learning_rate=0.1)

optimizer.zero_grad()
loss = w.square().sum()
loss.backward()

print("Gradient:", w.grad.item())

optimizer.step()
print("Updated weight:", w.item())

Gradient: 4.0
Updated weight: 1.899999976158142


In [26]:
# Original training dataset: no random augmentation
train_dataset = datasets.CIFAR10(
    root=data_root,
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

# Fixed train/validation split
train_subset, val_subset = random_split(
    train_dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

# Separate dataset object so augmentation doesn't affect validation
augmented_dataset = copy(train_dataset)
augmented_dataset.transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
])

augmented_train_subset = Subset(
    augmented_dataset,
    train_subset.indices
)

# Separate test split: no random augmentation
test_dataset = datasets.CIFAR10(
    root=data_root,
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

# Use this loader for parameter updates
augmented_train_loader = DataLoader(
    augmented_train_subset,
    batch_size=batch_size,
    shuffle=True
)

# Use these loaders for evaluation
train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print(f"Training:   {len(train_subset):,}")
print(f"Validation: {len(val_subset):,}")
print(f"Test:       {len(test_dataset):,}")

Training:   45,000
Validation: 5,000
Test:       10,000


In [30]:
optimizer = Adam(model.parameters(), learning_rate=0.001)

train_loss = train_one_epoch(
    model, augmented_train_loader, optimizer, device
)

val_metrics = evaluate(model, val_loader, device)

print(f"Training loss: {train_loss:.4f}")
print(f"Validation loss: {val_metrics['loss']:.4f}")
print(f"Validation accuracy: {val_metrics['accuracy']:.2f}%")

Training loss: 1.5698
Validation loss: 1.3379
Validation accuracy: 53.80%


In [31]:
train_metrics = evaluate(model, train_loader, device)

print(f"Training evaluation loss: {train_metrics['loss']:.4f}")
print(f"Training evaluation accuracy: {train_metrics['accuracy']:.2f}%")

Training evaluation loss: 1.3094
Training evaluation accuracy: 53.70%
